# Práctica 4: Soluciones de los ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

### Ejercicio 1

La biblioteca `Unified Planning` permite leer un dominio de planificación automática a partir de un fichero PDDL y crear posteriormente instancias de ese dominio mediante la interfaz proporcionada por la biblioteca.

En este ejercicio se pretende seguir esa metodología para crear instancias del mundo de los bloques que contengan los bloques $B_{0}$ a $B_{N - 1}$, apilados inicialmente en ese orden, y en la que el objetivo sea que estén apilados al contrario, con el bloque $B_{N - 1}$ sobre la mesa, el bloque $B_{N - 2}$ sobre el $B_{N - 1}$, el $B_{N - 3}$ sobre el $B_{N - 2}$, etc.

Se pide realizar lo siguiente:

1. Leer el dominio del mundo de los bloques a partir del fichero `dominio_mundo_bloques.pddl`.

In [42]:
from unified_planning.io import PDDLReader

In [43]:
lector_PDDL = PDDLReader()
dominio_mundo_bloques = lector_PDDL.parse_problem("dominio_mundo_bloques.pddl", None)

In [44]:
print(dominio_mundo_bloques)

problem name = dominio_mundo_bloques

types = [object]

fluents = [
  bool sobre_la_mesa[b=object]
  bool sobre[b1=object, b2=object]
  bool agarrado[b=object]
  bool brazo_libre
  bool despejado[b=object]
]

actions = [
  action agarrar(object b) {
    preconditions = [
      (sobre_la_mesa(b) and despejado(b) and brazo_libre)
    ]
    effects = [
      sobre_la_mesa(b) := false
      despejado(b) := false
      brazo_libre := false
      agarrado(b) := true
    ]
  }
  action bajar(object b) {
    preconditions = [
      agarrado(b)
    ]
    effects = [
      agarrado(b) := false
      sobre_la_mesa(b) := true
      despejado(b) := true
      brazo_libre := true
    ]
  }
  action desapilar(object b1, object b2) {
    preconditions = [
      (sobre(b1, b2) and despejado(b1) and brazo_libre)
    ]
    effects = [
      sobre(b1, b2) := false
      despejado(b1) := false
      brazo_libre := false
      agarrado(b1) := true
      despejado(b2) := true
    ]
  }
  action apilar(object

2. Completar la definición de la función `crea_instancia_mundo_bloques`, sustituyendo los `...` por código adecuado para que, dado el número `N` de bloques, la función proporcione el problema del mundo de los bloques descrito anteriormente.

In [45]:
from unified_planning.shortcuts import *

In [46]:
def crea_instancia_mundo_bloques(N):
    instancia = dominio_mundo_bloques.clone()  # Trabajamos con una copia del dominio
    # Se añaden los objetos de la instancia
    tipo_objeto = instancia.user_type('object')
    bloques = [Object(f'B{i}', tipo_objeto) for i in range(N)]
    instancia.add_objects(bloques)
    # Se establece el estado inicial de la instancia
    sobre_la_mesa = dominio_mundo_bloques.fluent('sobre_la_mesa')
    sobre = dominio_mundo_bloques.fluent('sobre')
    agarrado = dominio_mundo_bloques.fluent('agarrado')
    brazo_libre = dominio_mundo_bloques.fluent('brazo_libre')
    despejado = dominio_mundo_bloques.fluent('despejado')
    instancia.set_initial_value(sobre_la_mesa(bloques[0]), True)
    for i in range(1, N):
        instancia.set_initial_value(sobre(bloques[i], bloques[i - 1]), True)
    instancia.set_initial_value(brazo_libre, True)
    instancia.set_initial_value(despejado(bloques[N - 1]), True)
    # Se establece el objetivo de la instancia
    instancia.add_goal(sobre_la_mesa(bloques[N - 1]))
    for i in range(N - 1, 0, -1):
        instancia.add_goal(sobre(bloques[i - 1], bloques[i]))
    return instancia

3. Usar el planificador `Fast Downward` para tratar de resolver la instancia con el mayor número posible de bloques.

In [47]:
planificador = OneshotPlanner(name='fast-downward')

  *** Credits ***
  * In operation mode `OneshotPlanner` at line 1 of `/tmp/ipykernel_3662246/2882055873.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.



In [48]:
problema_mundo_bloques = crea_instancia_mundo_bloques(100)

In [49]:
resultado = planificador.solve(problema_mundo_bloques)
print(resultado)

status: SOLVED_SATISFICING
engine: Fast Downward
plan: SequentialPlan:
    desapilar(B99, B98)
    bajar(B99)
    desapilar(B98, B97)
    apilar(B98, B99)
    desapilar(B97, B96)
    apilar(B97, B98)
    desapilar(B96, B95)
    apilar(B96, B97)
    desapilar(B95, B94)
    apilar(B95, B96)
    desapilar(B94, B93)
    apilar(B94, B95)
    desapilar(B93, B92)
    apilar(B93, B94)
    desapilar(B92, B91)
    apilar(B92, B93)
    desapilar(B91, B90)
    apilar(B91, B92)
    desapilar(B90, B89)
    apilar(B90, B91)
    desapilar(B89, B88)
    apilar(B89, B90)
    desapilar(B88, B87)
    apilar(B88, B89)
    desapilar(B87, B86)
    apilar(B87, B88)
    desapilar(B86, B85)
    apilar(B86, B87)
    desapilar(B85, B84)
    apilar(B85, B86)
    desapilar(B84, B83)
    apilar(B84, B85)
    desapilar(B83, B82)
    apilar(B83, B84)
    desapilar(B82, B81)
    apilar(B82, B83)
    desapilar(B81, B80)
    apilar(B81, B82)
    desapilar(B80, B79)
    apilar(B80, B81)
    desapilar(B79, B78)
    apilar(

In [ ]:
if resultado.plan is not None:
    print(f"Plan encontrado con {len(resultado.plan.actions)} pasos.")

Plan encontrado con 200 pasos.


In [52]:
planificador = OneshotPlanner(name='fast-downward-opt')
resultado = planificador.solve(problema_mundo_bloques)
print(resultado)
if resultado.plan is not None:
    print(f"Plan encontrado con {len(resultado.plan.actions)} pasos.")

  *** Credits ***
  * In operation mode `OneshotPlanner` at line 1 of `/tmp/ipykernel_3662246/3370536230.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.

status: SOLVED_SATISFICING
engine: Fast Downward (with optimality guarantee)
plan: SequentialPlan:
    desapilar(B99, B98)
    bajar(B99)
    desapilar(B98, B97)
    apilar(B98, B99)
    desapilar(B97, B96)
    apilar(B97, B98)
    desapilar(B96, B95)
    apilar(B96, B97)
    desapilar(B95, B94)
    apilar(B95, B96)
    desapilar(B94, B93)
    apilar(B94, B95)
    desapilar(B93, B92)
    apilar(B93, B94)
    desapilar(B92, B91)
    apilar(B92, B93)
    desapilar(B91, B90)
    apilar(B91, B92)
    desapilar(B90, B89)
    apilar(B90, B91)
    desapilar(B89, B88)
    apilar(B89, B90)
    desapilar(B88, B87)
    apil

### Ejercicio 2

En el marco de la _Conferencia Internacional sobre Planificación Automática y Planificación Temporal_ ([International Conference on Automated Planning and
Scheduling, ICAPS](http://www.icaps-conference.org/)) se celebra, con periodicidad aproximadamente trienal, la _Competición Internacional de Planificación_ (https://www.icaps-conference.org/competitions/).

Esta competición tiene diferentes objetivos: realizar una comparación empírica del estado del arte de los sistemas de planificación; destacar desafíos para la comunidad de Planificación Automática; proponer nuevas direcciones para la investigación y nuevos vínculos con otros campos de la Inteligencia Artificial; y proporcionar nuevos conjuntos de datos que puedan ser utilizados por la comunidad científica como puntos de referencia.

Uno de los dominios utilizados en la competición del año 2002 combinaba el mundo de los bloques con la distribución logística de cajas. En este dominio hay una serie de camiones (que asumimos con capacidad infinita) que transportan cajas entre distintos lugares (que asumimos que están todos conectados entre sí); en esos lugares hay unos palés, sobre los que las cajas se colocan apiladas; los apilamientos se realizan con [polipastos](https://es.wikipedia.org/wiki/Polipasto) (hay al menos uno en cada lugar).

En este ejercicio se pide completar la especificación del dominio que se proporciona a continuación, sustituyendo en las acciones los `...` por hechos adecuados, y tratar de resolver la mayor cantidad posible de las instancias de problemas proporcionadas en la carpeta Depot.

In [53]:
from unified_planning.shortcuts import *

In [62]:
dominio_depot = Problem('Depot')

In [63]:
# Jerarquía de tipos de objetos

Place = UserType('Place')  # Lugar
Locatable = UserType('Locatable')  # Ubicable
Depot = UserType('Depot', Place)  # Almacén (Tipo de lugar)
Distributor = UserType('Distributor', Place)  # Distribuidor (Tipo de lugar)
Truck = UserType('Truck', Locatable)  # Camión (Tipo de ubicable)
Hoist = UserType('Hoist', Locatable)  # Polipasto (Tipo de ubicable)
Surface = UserType('Surface', Locatable)  # Superficie (Tipo de ubicable)
Pallet = UserType('Pallet', Surface)  # Palé (Tipo de superficie)
Crate = UserType('Crate', Surface)  # Caja (Tipo de superficie)

for tipo_de_objeto in [Place, Locatable, Depot, Distributor, Truck, Hoist, Surface, Pallet, Crate]:
    dominio_depot.user_types.append(tipo_de_objeto)

In [64]:
# Predicados booleanos

# El predicado AT representa que el ubicable x está en el lugar y
at = Fluent('AT', BoolType(), x=Locatable, y=Place)
# El predicado ON representa que la caja x está sobre la superficie y
on = Fluent('ON', BoolType(), x=Crate, y=Surface)
# El predicado IN representa que la caja x está en el camión y
# (Nótese el guión bajo incluido en el nombre de la variable, ya que no se
# puede usar in, al tratarse de un identificador reservado de Python)
in_ = Fluent('IN', BoolType(), x=Crate, y=Truck)
# El predicado LIFTING representa que el polipasto x está levantando la caja y
lifting = Fluent('LIFTING', BoolType(), x=Hoist, y=Crate)
# El predicado AVAILABLE representa que el polipasto x está disponible
available = Fluent('AVAILABLE', BoolType(), x=Hoist)
# El predicado CLEAR representa que la superficie x está despejada
clear = Fluent('CLEAR', BoolType(), x=Surface)

for fluente in [at, on, in_, lifting, available, clear]:
    dominio_depot.add_fluent(fluente, default_initial_value=False)

In [ ]:
# Esquemas de acciones

# La acción DRIVE representa que el camión x va del lugar y al lugar z
drive = InstantaneousAction('DRIVE', x=Truck, y=Place, z=Place)
x = drive.x
y = drive.y
z = drive.z
for hecho in [at(x, y)]:
    drive.add_precondition(hecho)
for hecho in [at(x, y)]:
    drive.add_effect(hecho, False)
for hecho in [at(x, z)]:
    drive.add_effect(hecho, True)

# La acción LIFT representa que el polipasto x levanta la caja y que se
# encontraba sobre la superficie z en el lugar p
lift = InstantaneousAction('LIFT', x=Hoist, y=Crate, z=Surface, p=Place)
x = lift.x
y = lift.y
z = lift.z
p = lift.p
for hecho in [at(x, p),
              available(x),
              at(y, p),
              on(y, z),
              clear(y)]:
    lift.add_precondition(hecho)
for hecho in [at(y, p),
              clear(y),
              available(x),
              on(y, z)]:
    lift.add_effect(hecho, False)
for hecho in [lifting(x, y),
              clear(z)]:
    lift.add_effect(hecho, True)

# La acción DROP representa que el polipasto x deja la caja y sobre la
# superficie z en el lugar p
drop = InstantaneousAction('DROP', x=Hoist, y=Crate, z=Surface, p=Place)
x = drop.x
y = drop.y
z = drop.z
p = drop.p
for hecho in [at(x, p),
              at(z, p),
              clear(z),
              lifting(x, y)]:
    drop.add_precondition(hecho)
for hecho in [lifting(x, y),
              clear(z)]:
    drop.add_effect(hecho, False)
for hecho in [available(x),
              at(y, p),
              clear(y),
              on(y, z)]:
    drop.add_effect(hecho, True)

# La acción LOAD representa que el polipasto x carga la caja y en el
# camión z en el lugar p
load = InstantaneousAction('LOAD', x=Hoist, y=Crate, z=Truck, p=Place)
x = load.x
y = load.y
z = load.z
p = load.p
for hecho in [at(x, p),
              at(z, p),
              lifting(x, y)]:
    load.add_precondition(hecho)
for hecho in [lifting(x, y)]:
    load.add_effect(hecho, False)
for hecho in [in_(y, z),
              available(x)]:
    load.add_effect(hecho, True)

# La acción UNLOAD representa que el polipasto x descarga la caja y del
# camión z en el lugar p
unload = InstantaneousAction('UNLOAD', x=Hoist, y=Crate, z=Truck, p=Place)
x = unload.x
y = unload.y
z = unload.z
p = unload.p
for hecho in [at(x, p),
              at(z, p),
              available(x),
              in_(y, z)]:
    unload.add_precondition(hecho)
for hecho in [in_(y, z),
              available(x)]:
    unload.add_effect(hecho, False)
for hecho in [lifting(x, y)]:
    unload.add_effect(hecho, True)

dominio_depot.add_actions([drive, lift, drop, load, unload])

AT(x, p)
AVAILABLE(x)
AT(y, p)
ON(y, z)
CLEAR(y)


In [ ]:
print(dominio_depot)

problem name = Depot

types = [Place, Locatable, Depot - Place, Distributor - Place, Truck - Locatable, Hoist - Locatable, Surface - Locatable, Pallet - Surface, Crate - Surface]

fluents = [
  bool AT[x=Locatable, y=Place]
  bool ON[x=Crate - Surface, y=Surface - Locatable]
  bool IN[x=Crate - Surface, y=Truck - Locatable]
  bool LIFTING[x=Hoist - Locatable, y=Crate - Surface]
  bool AVAILABLE[x=Hoist - Locatable]
  bool CLEAR[x=Surface - Locatable]
]

actions = [
  action DRIVE(Truck - Locatable x, Place y, Place z) {
    preconditions = [
      AT(x, y)
    ]
    effects = [
      AT(x, y) := false
      AT(x, z) := true
    ]
  }
  action LIFT(Hoist - Locatable x, Crate - Surface y, Surface - Locatable z, Place p) {
    preconditions = [
      AT(x, p)
      AVAILABLE(x)
      AT(y, p)
      ON(y, z)
      CLEAR(y)
    ]
    effects = [
      AT(y, p) := false
      CLEAR(y) := false
      AVAILABLE(x) := false
      ON(y, z) := false
      LIFTING(x, y) := true
      CLEAR(z) := t

In [15]:
from unified_planning.io import PDDLWriter

In [16]:
escritor_PDDL = PDDLWriter(dominio_depot)
escritor_PDDL.write_domain('dominio_depot.pddl')

In [ ]:
lector_PDDL = PDDLReader()
problema_depot = lector_PDDL.parse_problem('dominio_depot.pddl',
                                           'Depot/pfile15')

In [66]:
class PDDLWriter2:
    def __init__(self, domain_name, types, predicates, actions):
        self.domain_name = domain_name
        self.types = types
        self.predicates = predicates
        self.actions = actions
    
    def write_domain(self, filename):
        with open(filename, 'w') as f:
            # Cabecera
            f.write(f"(define\n  (domain {self.domain_name})\n")
            f.write("  (:requirements :strips :typing)\n")
            
            # Types
            f.write("  (:types\n")
            for type_def in self.types:
                f.write(f"    {type_def}\n")
            f.write("  )\n")
            
            # Predicates
            f.write("  (:predicates\n")
            for pred in self.predicates:
                f.write(f"    {pred}\n")
            f.write("  )\n")
            
            # Actions
            for action in self.actions:
                f.write(f"  (:action {action['name']}\n")
                f.write(f"    :parameters ({action['parameters']})\n")
                f.write("    :precondition (and\n")
                for cond in action['precondition']:
                    f.write(f"      {cond}\n")
                f.write("    )\n")
                f.write("    :effect (and\n")
                for eff in action['effect']:
                    f.write(f"      {eff}\n")
                f.write("    )\n")
                f.write("  )\n")
            
            f.write(")\n")

In [67]:
escritor_PDDL = PDDLWriter2(dominio_depot)
escritor_PDDL.write_domain('dominio_depot_2.pddl')

TypeError: PDDLWriter2.__init__() missing 3 required positional arguments: 'types', 'predicates', and 'actions'

In [18]:
planificador = OneshotPlanner(name='fast-downward')

  *** Credits ***
  * In operation mode `OneshotPlanner` at line 1 of `/tmp/ipykernel_3662246/2882055873.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.



In [19]:
resultado = planificador.solve(problema_depot)
print(resultado)

status: SOLVED_SATISFICING
engine: Fast Downward
plan: SequentialPlan:
    drive(truck0, distributor1, distributor2)
    lift(hoist5, crate11, crate5, distributor2)
    load(hoist5, crate11, truck1, distributor2)
    drive(truck1, distributor2, distributor1)
    lift(hoist4, crate12, crate10, distributor1)
    load(hoist4, crate12, truck1, distributor1)
    unload(hoist4, crate11, truck1, distributor1)
    drop(hoist4, crate11, crate10, distributor1)
    drive(truck1, distributor1, distributor2)
    unload(hoist5, crate12, truck1, distributor2)
    drive(truck1, distributor2, depot1)
    lift(hoist1, crate4, pallet6, depot1)
    drop(hoist1, crate4, crate7, depot1)
    drive(truck1, depot1, distributor0)
    lift(hoist3, crate9, pallet7, distributor0)
    load(hoist3, crate9, truck1, distributor0)
    drive(truck1, distributor0, depot0)
    drive(truck1, depot0, depot1)
    unload(hoist1, crate9, truck1, depot1)
    lift(hoist4, crate11, crate10, distributor1)
    drive(truck1, depot1,

### Ejercicio 3

[Sokoban](https://en.wikipedia.org/wiki/Sokoban) es un videojuego clásico de tipo puzle. En este juego el objetivo es empujar cajas, u otro tipo de objetos, en un almacén hasta llevarlos a las ubicaciones de almacenamiento. El juego se ve desde una perspectiva cenital. Los objetos solo se pueden empujar, nunca tirar de ellos, y solo un objeto se puede empujar a la vez. El desafío principal es planificar movimientos correctamente para evitar causar un punto muerto, una situación en la que un objeto o el jugador queda atrapado permanentemente, haciendo que el rompecabezas sea irresoluble.

En este ejercicio se pide lo siguiente:

1. Construir un dominio de planificación automática para el juego del Sokoban. Ese dominio debe contener los siguientes elementos:
   * Tipos de objetos: `thing`, `location`, `direction`, `player` (subtipo de `thing`), `stone` (subtipo de `thing`).
   * Predicados:
     * `CLEAR`: representa que una determinada localización (`location`) no contiene ninguna cosa (`thing`).
     * `AT`: representa que una cosa (`thing`) está en una determinada localización (`location`).
     * `AT-GOAL`: representa que una piedra (`stone`) está en una localización objetivo.
     * `IS-GOAL`: representa que una determinada localización (`location`) es una localización objetivo.
     * `IS-NONGOAL`: representa que una determinada localización (`location`) no es una localización objetivo.
     * `MOVE-DIR`: representa que se puede pasar de una determinada localización (`location`) a otra localización (`location`) adyacente moviéndose en una cierta dirección (`direction`).
   * Acciones:
     * `MOVE`: representa que el jugador (`player`) se mueve de la localización (`location`) que ocupa a una localización (`location`) libre adyacente en una determinada dirección (`direction`).
     * `PUSH-TO-NONGOAL`: representa que el jugador (`player`), estando en una determinada localización (`location`), empuja una piedra (`stone`) desde una localización (`location`) a otra localización (`location`) libre adyacente, que no es una localización objetivo, en una determinada dirección (`direction`).
     * `PUSH-TO-GOAL`: representa que el jugador (`player`), estando en una determinada localización (`location`), empuja una piedra (`stone`) desde una localización (`location`) a otra localización (`location`) libre adyacente, que es una localización objetivo, en una determinada dirección (`direction`).

In [20]:
from unified_planning.shortcuts import *

In [21]:
dominio_sokoban = Problem('Sokoban')

In [22]:
thing = UserType('thing')
location = UserType('location')
direction = UserType('direction')
player = UserType('player', father=thing)
stone = UserType('stone', father=thing)

In [23]:
for tipo_objeto in [thing, location, direction, player, stone]:
    dominio_sokoban.user_types.append(tipo_objeto)

In [24]:
clear = Fluent('CLEAR', BoolType(), l=location)
at = Fluent('AT', BoolType(), t=thing, l=location)
at_goal = Fluent('AT-GOAL', BoolType(), s=stone)
is_goal = Fluent('IS-GOAL', BoolType(), l=location)
is_nongoal = Fluent('IS-NONGOAL', BoolType(), l=location)
move_dir = Fluent('MOVE-DIR', BoolType(), f=location, t=location, d=direction)

In [25]:
for fluente in [clear, at, at_goal, is_goal, is_nongoal, move_dir]:
    dominio_sokoban.add_fluent(fluente, default_initial_value=False)

In [26]:
move = InstantaneousAction('MOVE', p=player, f=location, t=location, d=direction)
p = move.p
f = move.f
t = move.t
d = move.d
for hecho in [at(p, f),
              clear(t),
              move_dir(f, t, d)]:
    move.add_precondition(hecho)
for hecho in [at(p, f),
              clear(t)]:
    move.add_effect(hecho, False)
for hecho in [at(p, t),
              clear(f)]:
    move.add_effect(hecho, True)

In [27]:
push_to_nongoal = InstantaneousAction(
    'PUSH-TO-NONGOAL',
    p=player, pos=location, s=stone, f=location, t=location, d=direction)
p = push_to_nongoal.p
pos = push_to_nongoal.pos
s = push_to_nongoal.s
f = push_to_nongoal.f
t = push_to_nongoal.t
d = push_to_nongoal.d
for hecho in [at(p, pos),
              at(s, f),
              clear(t),
              move_dir(pos, f, d),
              move_dir(f, t, d),
              is_nongoal(t)]:
    push_to_nongoal.add_precondition(hecho)
for hecho in [at(p, pos),
              at(s, f),
              clear(t),
              at_goal(s)]:
    push_to_nongoal.add_effect(hecho, False)
for hecho in [at(p, f),
              at(s, t),
              clear(pos)]:
    push_to_nongoal.add_effect(hecho, True)

In [28]:
push_to_goal = InstantaneousAction(
    'PUSH-TO-GOAL',
    p=player, pos=location, s=stone, f=location, t=location, d=direction)
p = push_to_goal.p
pos = push_to_goal.pos
s = push_to_goal.s
f = push_to_goal.f
t = push_to_goal.t
d = push_to_goal.d
for hecho in [at(p, pos),
              at(s, f),
              clear(t),
              move_dir(pos, f, d),
              move_dir(f, t, d),
              is_goal(t)]:
    push_to_goal.add_precondition(hecho)
for hecho in [at(p, pos),
              at(s, f),
              clear(t)]:
    push_to_goal.add_effect(hecho, False)
for hecho in [at(p, f),
              at(s, t),
              clear(pos),
              at_goal(s)]:
    push_to_goal.add_effect(hecho, True)

In [29]:
dominio_sokoban.add_actions([move, push_to_nongoal, push_to_goal])

In [30]:
print(dominio_sokoban)

problem name = Sokoban

types = [thing, location, direction, player - thing, stone - thing]

fluents = [
  bool CLEAR[l=location]
  bool AT[t=thing, l=location]
  bool AT-GOAL[s=stone - thing]
  bool IS-GOAL[l=location]
  bool IS-NONGOAL[l=location]
  bool MOVE-DIR[f=location, t=location, d=direction]
]

actions = [
  action MOVE(player - thing p, location f, location t, direction d) {
    preconditions = [
      AT(p, f)
      CLEAR(t)
      MOVE-DIR(f, t, d)
    ]
    effects = [
      AT(p, f) := false
      CLEAR(t) := false
      AT(p, t) := true
      CLEAR(f) := true
    ]
  }
  action PUSH-TO-NONGOAL(player - thing p, location pos, stone - thing s, location f, location t, direction d) {
    preconditions = [
      AT(p, pos)
      AT(s, f)
      CLEAR(t)
      MOVE-DIR(pos, f, d)
      MOVE-DIR(f, t, d)
      IS-NONGOAL(t)
    ]
    effects = [
      AT(p, pos) := false
      AT(s, f) := false
      CLEAR(t) := false
      AT-GOAL(s) := false
      AT(p, f) := true
      AT(s, 

In [31]:
from unified_planning.io import PDDLWriter

In [32]:
escritor_PDDL = PDDLWriter(dominio_sokoban)
escritor_PDDL.write_domain('dominio_Sokoban.pddl')

2. Usar el algoritmo $\mathrm{A}^{*}$ y la heurística $h^{\mathrm{max}}$ para resolver, con la menor cantidad posible de movimientos de empuje, los puzles que se encuentran en la carpeta Sokoban. Para ello, asignar coste $1$ a las acciones `PUSH-TO-NONGOAL` y `PUSH-TO-GOAL` y coste $0$ al resto de acciones.

In [33]:
from unified_planning.io import PDDLReader

In [34]:
lector_PDDL = PDDLReader()
problema_Sokoban = lector_PDDL.parse_problem('dominio_Sokoban.pddl',
                                             'Sokoban/p01.pddl')

In [35]:
coste_de_empujar = Fluent('COSTE-DE-EMPUJAR', IntType(),
                          p=player, pos=location, s=stone, f=location, t=location, d=direction)
problema_Sokoban.add_fluent(coste_de_empujar, default_initial_value=Int(1))

integer COSTE-DE-EMPUJAR[p=player - thing, pos=location, s=stone - thing, f=location, t=location, d=direction]

In [36]:
métrica = MinimizeActionCosts(
    {push_to_nongoal: coste_de_empujar(
        push_to_nongoal.p, push_to_nongoal.pos, push_to_nongoal.s,
        push_to_nongoal.f, push_to_nongoal.t, push_to_nongoal.d),
     push_to_goal: coste_de_empujar(
        push_to_goal.p, push_to_goal.pos, push_to_goal.s,
        push_to_goal.f, push_to_goal.t, push_to_goal.d)},
    default=Int(0))

In [37]:
problema_Sokoban.add_quality_metric(métrica)

In [38]:
print(problema_Sokoban)

problem name = p01

types = [thing, location, direction, player - thing, stone - thing]

fluents = [
  bool clear[l=location]
  bool at[t=thing, l=location]
  bool at-goal[s=stone - thing]
  bool is-goal[l=location]
  bool is-nongoal[l=location]
  bool move-dir[f=location, t_0=location, d=direction]
  integer COSTE-DE-EMPUJAR[p=player - thing, pos=location, s=stone - thing, f=location, t=location, d=direction]
]

actions = [
  action move(player - thing p, location f, location t_0, direction d) {
    preconditions = [
      (at(p, f) and clear(t_0) and move-dir(f, t_0, d))
    ]
    effects = [
      at(p, f) := false
      clear(t_0) := false
      at(p, t_0) := true
      clear(f) := true
    ]
  }
  action push-to-nongoal(player - thing p, location pos, stone - thing s, location f, location t_0, direction d) {
    preconditions = [
      (at(p, pos) and at(s, f) and clear(t_0) and move-dir(pos, f, d) and move-dir(f, t_0, d) and is-nongoal(t_0))
    ]
    effects = [
      at(p, pos)

In [39]:
planificador = OneshotPlanner(name='fast-downward',
                              params={'fast_downward_search_config': 'astar(hmax())'})

  *** Credits ***
  * In operation mode `OneshotPlanner` at line 1 of `/tmp/ipykernel_3662246/3851894197.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.



In [40]:
resultado = planificador.solve(problema_Sokoban)
print(resultado)

status: SOLVED_SATISFICING
engine: Fast Downward
plan: SequentialPlan:
    move(player-01, pos-5-5, pos-5-4, dir-up)
    move(player-01, pos-5-4, pos-5-3, dir-up)
    move(player-01, pos-5-3, pos-4-3, dir-left)
    move(player-01, pos-4-3, pos-4-2, dir-up)
    move(player-01, pos-4-2, pos-3-2, dir-left)
    move(player-01, pos-3-2, pos-2-2, dir-left)
    move(player-01, pos-2-2, pos-2-3, dir-down)
    push-to-nongoal(player-01, pos-2-3, stone-01, pos-3-3, pos-4-3, dir-right)
    move(player-01, pos-3-3, pos-3-4, dir-down)
    push-to-nongoal(player-01, pos-3-4, stone-02, pos-4-4, pos-5-4, dir-right)
    move(player-01, pos-4-4, pos-3-4, dir-left)
    move(player-01, pos-3-4, pos-3-3, dir-up)
    move(player-01, pos-3-3, pos-3-2, dir-up)
    move(player-01, pos-3-2, pos-4-2, dir-right)
    push-to-nongoal(player-01, pos-4-2, stone-01, pos-4-3, pos-4-4, dir-down)
    move(player-01, pos-4-3, pos-5-3, dir-right)
    push-to-nongoal(player-01, pos-5-3, stone-02, pos-5-4, pos-5-5, dir-down)